# Day 049 — Exercise 2: overfitting_report

**What you'll build:** `overfitting_report(X, y, max_depths=range(1, 11), ...) -> pd.DataFrame` — train `DecisionTreeRegressor` at each depth, record train R² and test R², and flag rows where the gap exceeds 0.1 as overfitting.

**Why it matters:** The bias-variance tradeoff shows up clearly in decision trees: a depth-1 tree underfits (both scores are low); a depth-20 tree memorises the training data (train R² = 1.0) but generalises poorly. Plotting the gap as a function of depth is the classic *overfitting curve*.

## Provided: Setup + cross_validate_model

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }

## Your Implementation

In [ ]:
def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """
    Train DecisionTreeRegressor at each max_depth.
    Returns DataFrame with columns:
        max_depth, train_r2, test_r2, gap, overfit
    where gap = train_r2 - test_r2 and overfit = (gap > 0.1).
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        # TODO: m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        # TODO: m.fit(X_train, y_train)
        # TODO: tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        # TODO: te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        # TODO: gap   = round(tr_r2 - te_r2, 4)
        # TODO: records.append({
        #     'max_depth': depth,
        #     'train_r2':  round(tr_r2, 4),
        #     'test_r2':   round(te_r2, 4),
        #     'gap':       gap,
        #     'overfit':   bool(gap > 0.1),
        # })
        pass
    return pd.DataFrame(records)

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_regression_data(200)
    X  = df.drop(columns=['price'])
    y  = df['price']

    # Check 1: defined, returns DataFrame
    try:
        assert 'overfitting_report' in globals()
        report = overfitting_report(X, y)
        assert isinstance(report, pd.DataFrame), \
            f'expected DataFrame, got {type(report).__name__}'
        passed += 1; print('\u2705 Check 1: overfitting_report returns DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: required columns present
    try:
        for col in ('max_depth', 'train_r2', 'test_r2', 'gap', 'overfit'):
            assert col in report.columns, f'missing column: {col!r}'
        passed += 1; print(f'\u2705 Check 2: all 5 required columns present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: row count == 10 (default range(1, 11))
    try:
        assert len(report) == 10, \
            f'expected 10 rows (depths 1-10), got {len(report)}'
        passed += 1; print(f'\u2705 Check 3: 10 rows (one per depth)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: depth=1 shows underfitting — test_r2 < train_r2
    try:
        row1 = report[report['max_depth'] == 1].iloc[0]
        assert row1['test_r2'] < row1['train_r2'], \
            f'at depth=1, test_r2 should be < train_r2; ' \
            f'got train={row1["train_r2"]:.3f} test={row1["test_r2"]:.3f}'
        passed += 1; print(f'\u2705 Check 4: depth=1 train_r2={row1["train_r2"]:.3f} > test_r2={row1["test_r2"]:.3f}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: depth=10 memorises training data — train_r2 > 0.99
    try:
        row10 = report[report['max_depth'] == 10].iloc[0]
        assert row10['train_r2'] > 0.99, \
            f'at depth=10, tree should memorise train data (train_r2 > 0.99), ' \
            f'got {row10["train_r2"]}'
        passed += 1; print(f'\u2705 Check 5: depth=10 train_r2={row10["train_r2"]} > 0.99 (memorised)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """Train DecisionTreeRegressors at each depth; return train vs test R² table."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        gap   = round(tr_r2 - te_r2, 4)
        records.append({
            'max_depth': depth,
            'train_r2':  round(tr_r2, 4),
            'test_r2':   round(te_r2, 4),
            'gap':       gap,
            'overfit':   bool(gap > 0.1),
        })
    return pd.DataFrame(records)
```

</details>